# Futures Market Making

In this exercise, you will examine the exchange trade data to
approximate the market making opportunities, corresponding
risk, and consider how the approximation could be improved.

## 1. Edge given up by market takers

### 1a. November

Use CME trade data from Databento to approximate the P&L
available to market makers in November crude oil on
2026-09-24. That is, assume the trades with a buy or sell
aggressor are trading against a limit order from a
market maker. Consider the time period from 8:30 AM Chicago
until settlement. Assume the market makers start flat
at 8:30 AM to keep the approximation limited to this
time range.

You should compile a data set of trades with a time, side,
price, and quantity.
From that, determine total
P&L for the day, the largest open position intraday, and
the final open position at the end of the day.

Then decompose the total P&L for the day into
- edge: P&L generated by trading better than a theoretical
  price. In this case, use the mid-price from just before the trade.
- inventory: P&L generated by holding a position as the price moves.
Note that if one component of the decomposition is
computed, then the other can be computed as the difference from
the total.

### 1b. December

Redo 1a. for the December contract.

### 1c. November-December Spread

Redo 1a. for the November-December spread contract.

In [1]:
# Import and check venv
import datetime
import sys
from collections.abc import Iterable
from zoneinfo import ZoneInfo

import databento as db
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from finm37000 import (
    as_ct,
    disk_cache,
    get_databento_api_key,
    get_official_stats,
    temp_env,
)

pio.templates.default = "plotly_dark"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set3  # type: ignore[attr-defined]

sys.executable

'/home/dafuz/GitHub/finm37000/workspace/.venv/bin/python'

In [2]:
with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    client = db.Historical()

In [3]:
cme = "GLBX.MDP3"
tz_chicago = ZoneInfo("America/Chicago")
trade_date = datetime.date(2026, 9, 24)
nov = "CLX6"
dec = "CLZ6"
nov_dec = "CLX6-CLZ6"
symbols = (nov, dec, nov_dec)

# CL settles off the 13:28-13:30 CT window, so treat 13:30 as "settlement".
mm_start = pd.Timestamp.combine(trade_date, datetime.time(8, 30)).tz_localize(
    tz_chicago
)
mm_end = pd.Timestamp.combine(trade_date, datetime.time(13, 30)).tz_localize(tz_chicago)
# 1000 bbl per contract. The spread has no unit_of_measure_qty in the
# definitions, but 1 spread = 1 lot of each leg so the same multiplier applies.
contract_size = 1000

### Data

The intro notebook used the `trades` schema, but for the edge calculation I need
the mid right *before* each trade. Databento's `tbbo` schema is exactly that: every
trade with the top of book as it stood just before the trade.

Side convention (see the calendars notebook):
* `A`: sell aggressor hits the bid, so the resting market maker **buys**
* `B`: buy aggressor lifts the offer, so the market maker **sells**
* `N`: implied fills with no aggressor in this book. The question says to use
  trades with a buy or sell aggressor, so I drop these.

In [4]:
cache_scope = "futures_mm"


@disk_cache(scope=cache_scope)
def cached_tbbo(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="tbbo",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


@disk_cache(scope=cache_scope)
def cached_definitions(
    dataset: str,
    symbols: Iterable[str],
    date: datetime.date,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        symbols=symbols,
        start=date,
        schema="definition",
    ).to_df()


@disk_cache(scope=cache_scope)
def cached_stats(
    dataset: str,
    symbols: Iterable[str],
    start: pd.Timestamp | datetime.date,
    end: pd.Timestamp | datetime.date | None = None,
) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        schema="statistics",
        symbols=symbols,
        start=start,
        end=end,
    ).to_df()


tbbo = cached_tbbo(dataset=cme, symbols=symbols, start=mm_start, end=mm_end)
defs = cached_definitions(dataset=cme, symbols=symbols, date=trade_date)
raw_stats = cached_stats(
    dataset=cme,
    symbols=symbols,
    start=trade_date,
    end=trade_date + datetime.timedelta(days=2),
)

In [5]:
defs[
    [
        "raw_symbol",
        "instrument_class",
        "expiration",
        "min_price_increment",
        "unit_of_measure_qty",
        "match_algorithm",
    ]
].drop_duplicates()

,raw_symbol,instrument_class,expiration,min_price_increment,unit_of_measure_qty,match_algorithm
ts_recv,,,,,,
2026-09-24 00:00:00+00:00,CLX6-CLZ6,S,2026-10-20 18:30:00+00:00,0.01,NaN,F
2026-09-24 00:00:00+00:00,CLX6,F,2026-10-20 18:30:00+00:00,0.01,1000.0,F
2026-09-24 00:00:00+00:00,CLZ6,F,2026-11-20 19:30:00+00:00,0.01,1000.0,F


All three books are `F` = FIFO matching. That will matter in question 3.

Settlements for the day, using `get_official_stats` from the intro notebook:

In [6]:
stats = get_official_stats(raw_stats, defs)
settles = stats.loc[trade_date, "Settlement price"]
settles

Symbol
CLX6         94.61
CLX6-CLZ6     3.71
CLZ6         90.90
Name: Settlement price, dtype: float64

In [7]:
# sanity check: the spread settle should be Nov - Dec
settles[nov] - settles[dec], settles[nov_dec]

(np.float64(3.7099999999999937), np.float64(3.71))

One more check before using `bid_px_00`/`ask_px_00` as "the mid before the trade".
An aggressive order that sweeps several levels shows up as several trade rows with
the same `sequence`. If the BBO were updated between those rows, the later fills
would be compared to a mid that already reflects the sweep. Looks like it isn't:
every row in a matching event carries the same pre-event BBO.

In [8]:
tbbo.groupby(["symbol", "sequence"])[["bid_px_00", "ask_px_00"]].nunique().max()

bid_px_00    1
ask_px_00    1
dtype: int64

In [9]:
tbbo.groupby(["symbol", "side"])["size"].agg(["count", "sum"])

count    sum
symbol    side              
CLX6      A     37808  74202
          B     38612  73976
          N      9147  12983
CLX6-CLZ6 A      7353  20958
          B      5848  21917
          N      6297  11260
CLZ6      A      8733  12940
          B      8850  13078
          N      6561   9577

### Market maker trades and P&L decomposition

For each trade the market maker quantity is $q_i$ (+ buy, - sell), price $p_i$,
mid before the trade $m_i$, and $S$ the settlement price. Starting flat,

$$
\text{Total} = \sum_i q_i (S - p_i)
= \underbrace{\sum_i q_i (m_i - p_i)}_{\text{edge}}
+ \underbrace{\sum_i q_i (S - m_i)}_{\text{inventory}}
$$

times the 1000 bbl multiplier. The inventory term is what you make or lose from
holding the position after each fill while the price moves toward the settle.
I compute the total and edge directly and inventory as the difference (and check it
against the direct formula once).

In [10]:
def make_mm_trades(tbbo: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """Aggressor trades in `symbol` from the resting (market maker) side."""
    df = tbbo[(tbbo["symbol"] == symbol) & tbbo["side"].isin(["A", "B"])].copy()
    df["time"] = as_ct(df["ts_event"])
    # size is uint, cast before flipping signs
    df["quantity"] = np.where(df["side"] == "A", 1, -1) * df["size"].astype(int)
    df["mid"] = (df["bid_px_00"] + df["ask_px_00"]) / 2
    df["position"] = df["quantity"].cumsum()
    cols = ["side", "price", "size", "quantity", "mid", "position"]
    return df.set_index("time")[cols]


def summarize_mm(
    trades: pd.DataFrame, settle: float, multiplier: float = contract_size
) -> pd.Series:
    """Total, edge, inventory P&L and position stats for one book."""
    q = trades["quantity"]
    p = trades["price"]
    position = trades["position"]
    total = ((settle - p) * q).sum() * multiplier
    edge = ((trades["mid"] - p) * q).sum() * multiplier
    volume = trades["size"].sum()
    return pd.Series(
        {
            "Trades": len(trades),
            "Volume": volume,
            "Total P&L": total,
            "Edge": edge,
            "Inventory": total - edge,
            "Edge/contract": edge / volume,
            "Max long": position.max(),
            "Max short": position.min(),
            "Largest open position": position.iloc[position.abs().argmax()],
            "Time of largest": position.index[position.abs().argmax()].strftime(
                "%H:%M:%S"
            ),
            "Final position": position.iloc[-1],
        }
    )


def plot_mm(trades: pd.DataFrame, symbol: str, rule: str = "1min") -> None:
    """Price, position, and P&L marked to mid, sampled every `rule` for plotting."""
    df = trades.copy()
    cash = -(df["quantity"] * df["price"]).cumsum()
    df["P&L at mid"] = (cash + df["position"] * df["mid"]) * contract_size
    df["Cumulative edge"] = (
        (df["mid"] - df["price"]) * df["quantity"]
    ).cumsum() * contract_size
    # 70k+ points per trace makes the notebook huge, downsample for display
    sampled = df.resample(rule).last().dropna()

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=["Mid", "Market maker position", "P&L ($)"],
    )
    fig.add_trace(go.Scatter(x=sampled.index, y=sampled["mid"], name="mid"), 1, 1)
    fig.add_trace(
        go.Scatter(x=sampled.index, y=sampled["position"], name="position"), 2, 1
    )
    for col in ("P&L at mid", "Cumulative edge"):
        fig.add_trace(go.Scatter(x=sampled.index, y=sampled[col], name=col), 3, 1)
    fig.update_layout(
        height=700,
        width=700,
        title_text=f"{symbol} market maker approximation {trade_date}",
    )
    fig.show()

### 1a. November (`CLX6`)

In [11]:
nov_trades = make_mm_trades(tbbo, nov)
nov_trades.head(10)

,side,price,size,quantity,mid,position
time,,,,,,
2026-09-24 08:30:00.001223637-05:00,B,93.95,4,-4,93.945,-4
2026-09-24 08:30:00.002394091-05:00,A,93.94,2,2,93.950,-2
2026-09-24 08:30:00.043655889-05:00,A,93.95,2,2,93.955,0
2026-09-24 08:30:00.047908773-05:00,A,93.95,3,3,93.955,3
2026-09-24 08:30:00.064976979-05:00,B,93.95,1,-1,93.945,2
2026-09-24 08:30:00.072118089-05:00,A,93.95,1,1,93.955,3
2026-09-24 08:30:00.074123977-05:00,A,93.95,1,1,93.955,4
2026-09-24 08:30:00.091812337-05:00,B,93.95,1,-1,93.945,3
2026-09-24 08:30:00.106110029-05:00,A,93.94,1,1,93.945,4


In [12]:
nov_summary = summarize_mm(nov_trades, settles[nov])
nov_summary

Trades                       76420
Volume                      148178
Total P&L                   7790.0
Edge                     1263245.0
Inventory               -1255455.0
Edge/contract             8.525186
Max long                       940
Max short                    -1886
Largest open position        -1886
Time of largest           11:04:17
Final position                 226
dtype: object

In [13]:
# check inventory against the direct formula
direct_inventory = (
    (settles[nov] - nov_trades["mid"]) * nov_trades["quantity"]
).sum() * contract_size
np.isclose(direct_inventory, nov_summary["Inventory"])

np.True_

In [14]:
plot_mm(nov_trades, nov)

**November results.**
The resting side of about 148k aggressor lots collected roughly \$1.26M of edge,
about \$8.50 per contract. Half a tick is only \$5, so the extra comes from
times when the book was wider than one tick and from aggressive orders that swept
more than one level.

Almost all of that went back out as inventory P&L (about -\$1.26M), so the total for
the day is only about **+\$7.8k**. The plot shows how it happened. Nov rallied from
about 94 to almost 96.8 between 9:00 and 11:00, and the takers were net buyers on the way up, so
the resting side ended up **short 1,886 lots at 11:04**. Marked to mid, that was about
-\$1.7M at the worst point. Price then fell more than \$2 in half an hour, the
short got bought back, and P&L recovered to around zero. The **final position is
long 226** at the 94.61 settle.

So the edge number is large and steady (the cumulative edge line just ramps up), but
whether the day ends up positive depends almost entirely on where price goes
while the position is open.

### 1b. December (`CLZ6`)

In [15]:
dec_trades = make_mm_trades(tbbo, dec)
dec_summary = summarize_mm(dec_trades, settles[dec])
dec_summary

Trades                      17583
Volume                      26018
Total P&L               -154520.0
Edge                     193155.0
Inventory               -347675.0
Edge/contract            7.423899
Max long                      343
Max short                    -173
Largest open position         343
Time of largest          11:15:44
Final position               -138
dtype: object

In [16]:
plot_mm(dec_trades, dec)

**December results.** Dec trades about 1/6 of the Nov volume. Edge is about
\$193k (\$7.40/contract), inventory about -\$348k, so **total about -\$155k**.
The **largest position is long 343 at 11:15**, and the **final position is short 138**.

The interesting thing is that the Dec position went the *opposite* way from Nov:
the resting side got long Dec during the rally (it made money until about 11:00), then
gave it back in the selloff. Short Nov plus long Dec is the same as being short the
Nov-Dec spread, and that is exactly what the next part shows in the spread book itself.

### 1c. November-December spread (`CLX6-CLZ6`)

Same convention as in the calendars notebook: buying `CLX6-CLZ6` means buying
Nov and selling Dec, price = Nov - Dec. P&L per spread lot is still
1000 x (change in spread price).

In [17]:
spread_trades = make_mm_trades(tbbo, nov_dec)
spread_summary = summarize_mm(spread_trades, settles[nov_dec])
spread_summary

Trades                      13201
Volume                      42875
Total P&L               -667690.0
Edge                     272975.0
Inventory               -940665.0
Edge/contract            6.366764
Max long                      370
Max short                   -2168
Largest open position       -2168
Time of largest          12:38:42
Final position               -959
dtype: object

In [18]:
plot_mm(spread_trades, nov_dec)

**Nov-Dec spread results.** This is where the market makers lost most of their money.
The spread went up nearly all day, from about 3.16 to 3.75 (settle 3.71), and the takers were
net buyers of 959 spreads. The resting side was short the whole time, reaching
**short 2,168 at 12:38**, and finished **short 959**.
Edge was about \$273k (\$6.37/lot, the spread is usually
one tick wide), but inventory was about -\$941k, giving a **total of about -\$668k**.

This looks like one-directional flow rather than noise. 38% of the spread volume came in
trades of 10+ lots, compared with 14% in Nov and 5% in Dec. With Nov the new front month,
my guess is that this is roll-type flow (e.g. shorts rolling Nov to Dec by buying the spread),
but I can't verify that from this data alone. Either way, a market maker who just
keeps taking the other side of persistent flow pays for it in inventory.

### Summary of 1

In [19]:
summary = pd.DataFrame({nov: nov_summary, dec: dec_summary, nov_dec: spread_summary})
summary

,CLX6,CLZ6,CLX6-CLZ6
Trades,76420,17583,13201
Volume,148178,26018,42875
Total P&L,7790.0,-154520.0,-667690.0
Edge,1263245.0,193155.0,272975.0
Inventory,-1255455.0,-347675.0,-940665.0
Edge/contract,8.525186,7.423899,6.366764
Max long,940,343,370
Max short,-1886,-173,-2168
Largest open position,-1886,343,-2168
Time of largest,11:04:17,11:15:44,12:38:42


Across the three books the edge adds up to about \$1.73M, but inventory losses are about
\$2.54M, so the "market making P&L" for the day is roughly **-\$815k**. In all three
cases the aggressors were right about direction (buying Nov, buying the Nov-Dec spread),
which is adverse selection in its simplest form. I would not draw a general conclusion from
one day, but it does show that the edge is the dependable part and inventory is the
risky part.

## 2. Margining

How much margin is required to maintain the end-of-day positions in problem 1.
Answer this for each contract individually, then as a combined position.
Exact requirements can be found here
https://www.cmegroup.com/markets/energy/crude-oil/light-sweet-crude.margins.html#marginsTab=OUTRIGHT&pageNumber=1
but for simplicity, let's
call it $\$9000$ for November (long or short) per contract,
$\$8000$ for December, and $\$2000$ for the Nov-Dec spread.

In [20]:
margin_per_contract = pd.Series({nov: 9_000, dec: 8_000, nov_dec: 2_000})
final_positions = summary.loc["Final position"].astype(int)
individual_margin = final_positions.abs() * margin_per_contract
pd.DataFrame({"Final position": final_positions, "Margin": individual_margin})

,Final position,Margin
CLX6,226,2034000
CLZ6,-138,1104000
CLX6-CLZ6,-959,1918000


In [21]:
individual_margin.sum()

np.int64(5056000)

For the combined position, break the spread back into legs, then pair up any
opposite Nov/Dec exposure as spreads (charged \$2000) and charge the leftover as
outrights.

In [22]:
spread_pos = final_positions[nov_dec]
net_legs = pd.Series(
    {
        nov: final_positions[nov] + spread_pos,
        dec: final_positions[dec] - spread_pos,
    }
)
net_legs

CLX6   -733
CLZ6    821
dtype: int64

In [23]:
def combined_margin(net_legs: pd.Series) -> pd.Series:
    """Margin with opposite Nov/Dec exposure paired off as spreads."""
    n_nov, n_dec = net_legs[nov], net_legs[dec]
    # same direction legs get no spread offset
    opposite = np.sign(n_nov) == -np.sign(n_dec)
    n_spreads = min(abs(n_nov), abs(n_dec)) if opposite else 0
    left_nov = abs(n_nov) - n_spreads
    left_dec = abs(n_dec) - n_spreads
    return pd.Series(
        {
            f"{nov_dec} x{n_spreads}": n_spreads * margin_per_contract[nov_dec],
            f"{nov} x{left_nov}": left_nov * margin_per_contract[nov],
            f"{dec} x{left_dec}": left_dec * margin_per_contract[dec],
        }
    )


combined = combined_margin(net_legs)
combined, combined.sum()

(CLX6-CLZ6 x733    1466000
 CLX6 x0                 0
 CLZ6 x88           704000
 dtype: int64,
 np.int64(2170000))

**Individually**, margin for the end-of-day positions is
* Nov: 226 x \$9,000 = \$2,034,000
* Dec: 138 x \$8,000 = \$1,104,000
* Nov-Dec: 959 x \$2,000 = \$1,918,000

for a total of **\$5,056,000**.

**Combined**, the short 959 spreads are short 959 Nov / long 959 Dec. Netting against the
outrights gives -733 Nov and +821 Dec. That is 733 Nov-Dec spreads (short) plus 88 long Dec
outright: 733 x \$2,000 + 88 x \$8,000 = **\$2,170,000**, less than half of the separate
number. If the clearing house did not recognize the spread offset at all, the same net legs
would need 733 x \$9,000 + 821 x \$8,000 = \$13.2M, so the spread credit is what keeps it
manageable.

In practice the CME uses SPAN, which gives spread credits across all months in a portfolio,
so a real calculation would differ, but the conclusion holds: risk across the curve should be
margined as one portfolio, not book by book.

## 3. Estimating Opportunity

Your boss says, "Let's join the other market makers,
quoting 10\% of their quantities, then we'll capture 1/11 of the
market making P&L in question 1". Critique this assessment.
Address the ways in which the calculations in 1 are
both over and under estimates.
Then provide a brief proposal for how you could provide
a more accurate simulation to better estimate
the opportunity in this market.

### Answer

The "1/11" comes from: if everyone else rests quantity $Q$ and we add $0.1Q$, we are
$0.1/1.1 = 1/11$ of the liquidity, so we take 1/11 of the fills and 1/11 of the P&L. That only
holds if fills are allocated pro rata and nothing else changes. Neither is true here, and
the starting number from 1 is itself shaky. On 2026-09-24 the total from 1 was about
-\$815k, so 1/11 of it would be a *loss* of about \$74k. Clearly the boss means something
like "1/11 of the edge", which is more like \$157k/day, and that is where most of my
objections apply.

**Why 1 overestimates the opportunity**
* *Not all resting orders belong to market makers.* Hedgers, CTAs, and execution algos also
  rest passive orders. Treating every passive fill as market maker P&L puts all of the edge
  into the market making pool, so it is too large.
* *FIFO matching.* The definitions show `match_algorithm = F` for all three books. Under FIFO,
  joining a price level means going to the back of the queue. We only get filled after
  everyone ahead of us, which tends to happen when a level gets taken out completely, and that is
  exactly when price is about to move against us. Our fills would be a smaller and *worse*
  subset than 1/11 of the total.
* *Latency / queue position.* Established market makers have better queue position and react
  faster when they need to pull quotes before a move. A new entrant gets more of the toxic fills.
* *Costs ignored.* Exchange and clearing fees are charged per contract. With an edge of only
  \$6-9/contract, fees take a meaningful share of it. Funding the margin from Q2 also costs money.
* *Sweeps inflate edge.* Fills several levels deep count as big edge relative to the
  pre-trade mid, but those deep orders are often not market makers quoting at the touch.
* *Mid as fair value.* The mid right before the trade ignores the information in the
  trade itself. Using a markout (the mid 1s, 10s, or 60s later) would usually give a lower edge.

**Why 1 underestimates the opportunity**
* *No risk management.* The calculation assumes a market maker that never skews quotes,
  never hedges, and just absorbs whatever arrives. A real market maker short 1,800 Nov would
  lean its quotes, hedge in Dec / the spread / Brent, or cut size. The -\$2.5M inventory loss
  is mostly the result of that passive behavior, not something every market maker has to accept.
* *Cross-book hedges.* Nov, Dec, and the spread were all treated separately, but the resting
  side was short Nov, long Dec, and short the spread, which are correlated positions. Managing them
  together, as the margin calculation in Q2 does, reduces risk.
* *Implied fills dropped.* The `N` trades (about 13k Nov, 9.6k Dec, 11k spread lots) were excluded,
  yet resting orders do get filled through implied matching, so some fills are missing.
* *Incomplete volume.* As the intro notebook showed, cleared volume is well above what shows up in
  the electronic trades record (blocks, other spreads implying into the legs). Only 8:30 to settle was
  used, so the overnight session is missing too.
* *Incentives.* CME has market maker / incentive programs for energy that reduce fees, which are
  not included here.

**Other problems with the plan**
* *One day.* Inventory P&L is basically one draw of how the price moved that day. A day with no
  trend would look very profitable; a trending day like this one looks terrible. Plan on at least
  a few months of days before quoting an expected value.
* *We change the market.* Adding 10% more size can change how other market makers quote, and
  our own trading affects the price. The trade data alone can't show that effect.
* *Capacity / risk.* Even 1/11 of these positions (around 170 Nov or 200 spreads at the peak) needs
  real capital and risk limits, which the boss's estimate does not mention.

**Proposal for a better simulation**
1. Use market-by-order (`mbo`) data instead of `tbbo`, rebuild the full order book, and keep track of
   queue position at each price level.
2. Insert our hypothetical orders into the book with a realistic latency for submit and cancel, and
   fill them under FIFO rules only when the volume traded at that level reaches our place in the queue.
   Check the matching rules per instrument (`match_algorithm`) and handle implied matching for the spread.
3. Define an actual quoting strategy: quote size, when to join or pull, inventory limits, quote skewing,
   and hedging between Nov, Dec, and the spread, then run that strategy instead of absorbing every fill.
4. Include fees, margin funding cost (using the portfolio margin from Q2), and use markouts at several
   horizons to measure adverse selection separately from end-of-day inventory P&L.
5. Run it over many days (trending, quiet, roll period, expiration) and report the distribution of daily P&L,
   drawdown, and margin used, not one number. If possible, test with a small live quote to calibrate
   our fill rate against the simulation.